In [1]:
#Apply Raw
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time
from sklearn.cluster import OPTICS
from sklearn.cluster import Birch

In [2]:
#load dataset
df= pd.read_csv("data/pain_dataset_200P_4hz.csv")
df

,person_ID,acc_x,acc_y,acc_z,eda,bvp,hr,temp,pain_scale
0,P001,0.2751,-0.0464,0.3049,0.7395,99.24,67.6,33.94,5
1,P001,0.2428,-0.1161,0.3641,0.7793,103.24,68.3,33.95,5
2,P001,0.0146,-0.1479,0.6552,0.8581,103.08,68.1,33.91,5
3,P001,-0.0806,-0.2144,0.6631,0.8881,104.12,66.6,33.94,5
4,P001,-0.0808,-0.1754,0.5448,0.7786,107.05,66.8,33.95,5
...,...,...,...,...,...,...,...,...,...
95995,P200,0.3618,0.0199,0.1452,5.2451,124.71,104.4,36.09,7
95996,P200,0.2842,-0.1367,0.0158,5.3054,125.71,103.4,36.09,7
95997,P200,0.3005,-0.1288,-0.1729,5.2345,125.07,103.3,36.13,7
95998,P200,0.2964,-0.1015,-0.2274,5.2046,126.12,102.7,36.19,7


In [3]:
# Drop target and ID column & target column
X_raw = df.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape (raw version):", X_raw.shape)


#Define Cluster Parameters
k_values = range(2, 9)  # clusters 2–8 for KMeans, GMM, Agglomerative, Spectral
n_init = 10  # random initialization for KMeans, GMM, Spectral
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

Features shape (raw version): (96000, 7)


In [6]:
#K-Means
start_time = time.time()
kmeans_raw = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    kmeans.fit(X_raw)
    labels = kmeans.labels_
    sil, db, ch = compute_metrics(X_raw, labels) #K-Means on Raw Featureslabels)
    kmeans_raw.append({"algorithm":"K-Means","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")

Runtime: 1032.8031475543976 seconds
K-Means runtime: 1032.8031 seconds


In [7]:
#GMM on Raw Features
start_time = time.time()
gmm_raw = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    gmm.fit(X_raw)
    labels = gmm.predict(X_raw)
    sil, db, ch = compute_metrics(X_raw, labels)
    gmm_raw.append({"algorithm":"GMM","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")

Runtime: 1383.9885017871857 seconds
GMM runtime: 1383.9885 seconds


In [8]:
#Agglomerative Clustering on Scaled + PCA Data
df_small = df.sample(n=5000, random_state=42)
X_small_raw = df_small.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape (raw version):", X_small_raw.shape)
start_time = time.time()
agg_raw = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    agg.fit(X_small_raw)
    labels = agg.labels_
    sil, db, ch = compute_metrics(X_small_raw, labels)
    agg_raw.append({"algorithm":"Agglomerative","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Features shape (raw version): (5000, 7)
Runtime: 7.261616230010986 seconds
Agglomerative runtime: 7.2616 seconds


In [9]:
#Spectral Clustering
start_time = time.time()
spectral_raw = []
for k in k_values:
    spectral = SpectralClustering(n_clusters=k, affinity='nearest_neighbors', n_init=n_init, random_state=42)
    spectral.fit(X_small_raw)
    labels = spectral.labels_
    sil, db, ch = compute_metrics(X_small_raw, labels)
    spectral_raw.append({"algorithm":"Spectral","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")
print("Features shape (raw version):", X_small_raw.shape)

Runtime: 8.268547534942627 seconds
Spectral runtime: 8.2685 seconds
Features shape (raw version): (5000, 7)


In [10]:
#DBSCAN
start_time = time.time()
dbscan_raw = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    dbscan.fit(X_raw)
    labels = dbscan.labels_
    sil, db, ch = compute_metrics(X_raw, labels)
    dbscan_raw.append({"algorithm":"DBSCAN","preprocessing":"raw","eps":eps,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds") 

Runtime: 451.0267765522003 seconds
DBSCAN runtime: 451.0268 seconds


In [11]:
#Birch
start_time = time.time()
birch_raw = []
#threshold_values = [0.2, 0.5, 1.0, 1.5]
threshold_values = [1.5, 3.0, 5.0, 10.0]

for t in threshold_values:
    birch = Birch(n_clusters=None, branching_factor=200,threshold=t)
    labels = birch.fit_predict(X_raw)

    if len(set(labels)) > 1:
        sil, db, ch = compute_metrics(X_raw, labels)
        birch_raw.append({
            "algorithm": "BIRCH",
            "preprocessing": "raw",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Birch runtime: {runtime:.4f} seconds")
print("Features shape (raw version):", X_small_raw.shape)

Runtime: 594.057256937027 seconds
Birch runtime: 594.0573 seconds
Features shape (raw version): (5000, 7)


In [12]:
#OPTICS
start_time = time.time()
optics_raw = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_raw)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_raw, labels)
        optics_raw.append({
            "algorithm": "OPTICS",
            "preprocessing": "raw",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

Runtime: 11723.99243068695 seconds
Optics runtime: 11723.9924 seconds


In [ ]:
import csv 

pain_results_raw = (kmeans_raw+gmm_raw+agg_raw+spectral_raw+dbscan_raw+birch_raw + optics_raw)


keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]
with open('updated_data/pain_new_data/pain_raw.csv', 'w', newline='') as file:
#with open('updated_data/pain_data/pain_raw.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(pain_results_raw)

In [ ]:
# from sklearn.metrics import adjusted_rand_score
# from sklearn.utils import resample
# import numpy as np
# import pandas as pd

# # ARI stability analysis
# n_bootstrap = 100
# ari_results = []
# # Collect all parameter settings from your previous results
# all_configs = []

# for r in kmeans_raw:
#     all_configs.append(("K-Means", {"k": r["k"]}))

# for r in gmm_raw:
#     all_configs.append(("GMM", {"k": r["k"]}))

# for r in agg_raw:
#     all_configs.append(("Agglomerative", {"k": r["k"]}))

# for r in spectral_raw:
#     all_configs.append(("Spectral", {"k": r["k"]}))

# for r in dbscan_raw:
#     all_configs.append(("DBSCAN", {"eps": r["eps"]}))

# for r in birch_raw:
#     all_configs.append(("BIRCH", {"threshold": r["threshold"]}))

# for r in optics_raw:
#     all_configs.append(("OPTICS", {"min_samples": r["min_samples"]}))
# # helper function to fit a model and return labels 
# def fit_and_predict(name, params, X_data):

#     if name == "K-Means":
#         model = KMeans(n_clusters=params["k"], n_init=n_init, random_state=42)
#         labels = model.fit_predict(X_data)

#     elif name == "GMM":
#         model = GaussianMixture(n_components=params["k"], n_init=n_init, random_state=42)
#         labels = model.fit(X_data).predict(X_data)

#     elif name == "Agglomerative":
#         model = AgglomerativeClustering(n_clusters=params["k"], linkage='ward')
#         labels = model.fit_predict(X_data)

#     elif name == "Spectral":
#         model = SpectralClustering(
#             n_clusters=params["k"],
#             affinity='nearest_neighbors',
#             n_init=n_init,
#             random_state=42
#         )
#         labels = model.fit_predict(X_data)

#     elif name == "DBSCAN":
#         model = DBSCAN(eps=params["eps"], min_samples=min_samples)
#         labels = model.fit_predict(X_data)

#     elif name == "BIRCH":
#         model = Birch(n_clusters=None, threshold=params["threshold"])
#         labels = model.fit_predict(X_data)

#     elif name == "OPTICS":
#         model = OPTICS(min_samples=params["min_samples"], xi=0.05, n_jobs=-1)
#         labels = model.fit_predict(X_data)

#     else:
#         return None

#     return labels


# # Reuse all parameter configurations from previous section
# for algo_name, params in all_configs:

#     # reference clustering on full data
#     ref_labels = fit_and_predict(algo_name, params, X_raw)

#     if ref_labels is None:
#         continue

#     ari_scores = []
#     rng = np.random.RandomState(42)

#     for b in range(n_bootstrap):

#         # bootstrap sample with indices
#         indices = rng.choice(len(X_raw), size=len(X_raw), replace=True)
#         X_boot = X_raw.iloc[indices]

#         boot_labels = fit_and_predict(algo_name, params, X_boot)

#         if boot_labels is None:
#             continue

#         # compare only sampled observations
#         ref_subset = np.array(ref_labels)[indices]

#         # remove noise points for DBSCAN / OPTICS
#         mask = (boot_labels != -1) & (ref_subset != -1)

#         if np.sum(mask) < 2:
#             continue

#         ari = adjusted_rand_score(ref_subset[mask], np.array(boot_labels)[mask])
#         ari_scores.append(ari)

#     if len(ari_scores) > 0:
#         ari_results.append({
#             "algorithm": algo_name,
#             **params,
#             "ARI_mean": np.mean(ari_scores),
#             "ARI_std": np.std(ari_scores)
#         })


# # Summary table
# ari_df = pd.DataFrame(ari_results).round(4)

# print("\nBOOTSTRAP ARI STABILITY ")
# print(ari_df.to_string(index=False))

# # Top 3 most stable by ARI
# top3_ari = ari_df.nlargest(3, "ARI_mean")

# print("\nTOP 3 MOST STABLE BY ARI ")
# print(top3_ari.to_string(index=False))

MemoryError: Unable to allocate 34.3 GiB for an array with shape (4607952000,) and data type float64

In [ ]:
# #the top three by stability score
# ari_df["Stability Score"] = (1 - 2 * ari_df["ARI_std"]).clip(0, 1) #"eps", "min_samples", "threshold"

# top_3 = (
#     ari_df
#     .sort_values(["Stability Score", "ARI_mean"], ascending=[False, False])
#     .head(3)
#     .loc[:, ["algorithm", "k","ARI_mean", "ARI_std", "Stability Score"]]
# )

# print(top_3.round(3).to_string(index=False))

In [ ]:
# #the best three clustering results overall, sort primarily by ARI_mean
# top_3 = ari_df.sort_values(
#     ["ARI_mean", "Stability Score"],
#     ascending=[False, False]
# ).head(3)

# print(top_3.round(3).to_string(index=False))

In [ ]:
# ari_df.to_csv("updated_data/ARI_Score/pain_raw_ari.csv", index=False)

In [ ]:
from sklearn.metrics import adjusted_rand_score
import numpy as np
import pandas as pd


# ARI settings
n_bootstrap = 100
ari_results = []

# Reproducible bootstrap sampling
rng = np.random.RandomState(42)

# Helper function

def fit_and_predict(name, params, X_data):

    if name == "K-Means":
        model = KMeans(
            n_clusters=params["k"],
            n_init=n_init,
            random_state=42
        )
        labels = model.fit_predict(X_data)

    elif name == "GMM":
        model = GaussianMixture(
            n_components=params["k"],
            n_init=n_init,
            random_state=42
        )
        labels = model.fit_predict(X_data)

    elif name == "Agglomerative":
        model = AgglomerativeClustering(
            n_clusters=params["k"],
            linkage='ward'
        )
        labels = model.fit_predict(X_data)

    elif name == "Spectral":
        model = SpectralClustering(
            n_clusters=params["k"],
            affinity='nearest_neighbors',
            n_init=n_init,
            random_state=42
        )
        labels = model.fit_predict(X_data)

    elif name == "DBSCAN":
        model = DBSCAN(
            eps=params["eps"],
            min_samples=min_samples
        )
        labels = model.fit_predict(X_data)

    elif name == "BIRCH":
        model = Birch(
            n_clusters=None,
            threshold=params["threshold"]
        )
        labels = model.fit_predict(X_data)

    elif name == "OPTICS":
        model = OPTICS(
            min_samples=params["min_samples"],
            xi=0.05,
            n_jobs=-1
        )
        labels = model.fit_predict(X_data)

    else:
        return None

    return labels



# Function for bootstrap ARI

def bootstrap_ari(algo_name, params, X_data):

    print(f"Processing: {algo_name} {params}")

    # Reference clustering on full dataset
    ref_labels = fit_and_predict(
        algo_name,
        params,
        X_data
    )

    if ref_labels is None:
        return None

    ari_scores = []

    for b in range(n_bootstrap):

        # Bootstrap sample
        indices = rng.choice(
            len(X_data),
            size=len(X_data),
            replace=True
        )

        X_boot = X_data.iloc[indices]

        # Cluster bootstrap sample
        boot_labels = fit_and_predict(
            algo_name,
            params,
            X_boot
        )

        if boot_labels is None:
            continue

        # Reference labels for sampled observations
        ref_subset = np.asarray(ref_labels)[indices]

        # ARI
        ari = adjusted_rand_score(
            ref_subset,
            np.asarray(boot_labels)
        )

        ari_scores.append(ari)

    if len(ari_scores) == 0:
        return None

    return {
        "algorithm": algo_name,
        **params,
        "ARI_mean": np.mean(ari_scores),
        "ARI_std": np.std(ari_scores),
        "n_bootstrap": len(ari_scores)
    }

In [23]:
for r in kmeans_raw:

    result = bootstrap_ari(
        "K-Means",
        {"k": r["k"]},
        X_raw
    )

    if result is not None:
        ari_results.append(result)

print("K-Means completed.")

Processing: K-Means {'k': 2}
Processing: K-Means {'k': 3}
Processing: K-Means {'k': 4}
Processing: K-Means {'k': 5}
Processing: K-Means {'k': 6}
Processing: K-Means {'k': 7}
Processing: K-Means {'k': 8}
K-Means completed.


In [24]:
for r in gmm_raw:

    result = bootstrap_ari(
        "GMM",
        {"k": r["k"]},
        X_raw
    )

    if result is not None:
        ari_results.append(result)

print("GMM completed.")

Processing: GMM {'k': 2}
Processing: GMM {'k': 3}
Processing: GMM {'k': 4}
Processing: GMM {'k': 5}
Processing: GMM {'k': 6}
Processing: GMM {'k': 7}
Processing: GMM {'k': 8}
GMM completed.


In [25]:
for r in agg_raw:

    result = bootstrap_ari(
        "Agglomerative",
        {"k": r["k"]},
        X_small_raw
    )

    if result is not None:
        ari_results.append(result)

print("Agglomerative completed.")

Processing: Agglomerative {'k': 2}
Processing: Agglomerative {'k': 3}
Processing: Agglomerative {'k': 4}
Processing: Agglomerative {'k': 5}
Processing: Agglomerative {'k': 6}
Processing: Agglomerative {'k': 7}
Processing: Agglomerative {'k': 8}
Agglomerative completed.


In [26]:
for r in spectral_raw:

    result = bootstrap_ari(
        "Spectral",
        {"k": r["k"]},
        X_small_raw
    )

    if result is not None:
        ari_results.append(result)

print("Spectral completed.")

Processing: Spectral {'k': 2}
Processing: Spectral {'k': 3}
Processing: Spectral {'k': 4}
Processing: Spectral {'k': 5}


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Processing: Spectral {'k': 6}
Processing: Spectral {'k': 7}
Processing: Spectral {'k': 8}
Spectral completed.


In [27]:
for r in dbscan_raw:

    result = bootstrap_ari(
        "DBSCAN",
        {"eps": r["eps"]},
        X_raw
    )

    if result is not None:
        ari_results.append(result)

print("DBSCAN completed.")

Processing: DBSCAN {'eps': 0.5}
Processing: DBSCAN {'eps': 1.0}
Processing: DBSCAN {'eps': 1.5}
DBSCAN completed.


In [28]:
for r in birch_raw:

    result = bootstrap_ari(
        "BIRCH",
        {"threshold": r["threshold"]},
        X_raw
    )

    if result is not None:
        ari_results.append(result)

print("BIRCH completed.")

Processing: BIRCH {'threshold': 1.5}
Processing: BIRCH {'threshold': 3.0}
Processing: BIRCH {'threshold': 5.0}
Processing: BIRCH {'threshold': 10.0}
BIRCH completed.


In [ ]:
for r in optics_raw:

    result = bootstrap_ari(
        "OPTICS",
        {"min_samples": r["min_samples"]},
        X_raw
    )

    if result is not None:
        ari_results.append(result)

print("OPTICS completed.")

Processing: OPTICS {'min_samples': 3}


c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
c:\Users\shetu\Study\Hochschule_Schmalkalden

In [ ]:

# Final ARI summary


ari_df = pd.DataFrame(ari_results).round(4)

print("\n BOOTSTRAP ARI STABILITY ")
print(ari_df.to_string(index=False))


# Top 3 by ARI mean
top3_ari = ari_df.nlargest(
    3,
    "ARI_mean"
)

print("\n TOP 3 MOST STABLE BY ARI ")
print(top3_ari.to_string(index=False))


# Save results
ari_df.to_csv("updated_data/ARI_Score/pain_raw_ari.csv", index=False)


In [ ]:
#the top three by stability score
ari_df["Stability Score"] = (1 - 2 * ari_df["ARI_std"]).clip(0, 1) #"eps", "min_samples", "threshold"

top_3 = (
    ari_df
    .sort_values(["Stability Score", "ARI_mean"], ascending=[False, False])
    .head(3)
    .loc[:, ["algorithm", "k","ARI_mean", "ARI_std", "Stability Score"]]
)

print(top_3.round(3).to_string(index=False))

In [ ]:
#the best three clustering results overall, sort primarily by ARI_mean
top_3 = ari_df.sort_values(
    ["ARI_mean", "Stability Score"],
    ascending=[False, False]
).head(3)

print(top_3.round(3).to_string(index=False))

In [15]:
#  Combine all algorithm results 
all_results = (
    kmeans_raw +
    gmm_raw +
    agg_raw +
    spectral_raw +
    dbscan_raw +
    birch_raw +
    optics_raw
)

results_df = pd.DataFrame(all_results)

# Round metric values to 4 decimal places
metric_cols = ["silhouette", "davies_bouldin", "calinski_harabasz"]
results_df[metric_cols] = results_df[metric_cols].round(4)

# Columns that may exist depending on algorithm
possible_cols = ["algorithm", "k", "eps", "threshold", "min_samples", "n_clusters"]

def available_cols(df, metric):
    cols = [c for c in possible_cols if c in df.columns]
    cols.append(metric)
    return cols

#  Top 3 by Silhouette (higher is better) 
top3_sil = results_df.nlargest(3, "silhouette")

print("\nTOP 3 SILHOUETTE ")
print(top3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Top 3 by Davies-Bouldin (lower is better) 
top3_db = results_df.nsmallest(3, "davies_bouldin")

print("\n TOP 3 DAVIES-BOULDIN ")
print(top3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Top 3 by Calinski-Harabasz (higher is better) 
top3_ch = results_df.nlargest(3, "calinski_harabasz")

print("\n TOP 3 CALINSKI-HARABASZ ")
print(top3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))

#  Bottom 3 by Silhouette (lower is worse) 
bottom3_sil = results_df.nsmallest(3, "silhouette")

print("\n BOTTOM 3 SILHOUETTE ")
print(bottom3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Bottom 3 by Davies-Bouldin (higher is worse) 
bottom3_db = results_df.nlargest(3, "davies_bouldin")

print("\n BOTTOM 3 DAVIES-BOULDIN ")
print(bottom3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Bottom 3 by Calinski-Harabasz (lower is worse) 
bottom3_ch = results_df.nsmallest(3, "calinski_harabasz")

print("\n BOTTOM 3 CALINSKI-HARABASZ ")
print(bottom3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))


TOP 3 SILHOUETTE 
algorithm   k  eps  threshold  min_samples  n_clusters  silhouette
  K-Means 2.0  NaN        NaN          NaN         NaN      0.5355
 Spectral 2.0  NaN        NaN          NaN         NaN      0.5309
      GMM 2.0  NaN        NaN          NaN         NaN      0.5288

 TOP 3 DAVIES-BOULDIN 
algorithm   k  eps  threshold  min_samples  n_clusters  davies_bouldin
  K-Means 2.0  NaN        NaN          NaN         NaN          0.6611
 Spectral 2.0  NaN        NaN          NaN         NaN          0.6637
      GMM 2.0  NaN        NaN          NaN         NaN          0.6666

 TOP 3 CALINSKI-HARABASZ 
algorithm   k  eps  threshold  min_samples  n_clusters  calinski_harabasz
  K-Means 2.0  NaN        NaN          NaN         NaN        188461.2235
      GMM 2.0  NaN        NaN          NaN         NaN        183950.9758
  K-Means 3.0  NaN        NaN          NaN         NaN        165386.6588

 BOTTOM 3 SILHOUETTE 
algorithm   k  eps  threshold  min_samples  n_clusters  sil

In [16]:

# TOP 3 RESULTS FOR EACH ALGORITHM INDIVIDUALLY



all_algorithms = {
    "K-Means": kmeans_raw,
    "GMM": gmm_raw,
    "Agglomerative": agg_raw,
    "Spectral": spectral_raw,
    "DBSCAN": dbscan_raw,
    "BIRCH": birch_raw,
    "OPTICS": optics_raw
}



for algorithm, results in all_algorithms.items():

    if len(results) == 0:
        continue

    result_df = pd.DataFrame(results)


    # Round all validation values to 4 decimal places

    result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ] = result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ].round(4)



    print("\n")
    print("="*60)
    print(algorithm)
    print("="*60)




    # Select parameter column


    parameter_columns = [
        "k",
        "eps",
        "threshold",
        "min_samples"
    ]


    parameter = None

    for col in parameter_columns:
        if col in result_df.columns:
            parameter = col
            break



   
    # TOP 3 SILHOUETTE
   

    print("\nTop 3 Silhouette Score (Higher is better)")

    top_sil = result_df.nlargest(
        3,
        "silhouette"
    )

    print(
        top_sil[
            [
                parameter,
                "silhouette"
            ]
        ].to_string(index=False)
    )



   
    # TOP 3 DAVIES-BOULDIN
  
    print("\nTop 3 Davies-Bouldin Index (Lower is better)")

    top_db = result_df.nsmallest(
        3,
        "davies_bouldin"
    )

    print(
        top_db[
            [
                parameter,
                "davies_bouldin"
            ]
        ].to_string(index=False)
    )



  
    # TOP 3 CALINSKI-HARABASZ


    print("\nTop 3 Calinski-Harabasz Index (Higher is better)")

    top_ch = result_df.nlargest(
        3,
        "calinski_harabasz"
    )

    print(
        top_ch[
            [
                parameter,
                "calinski_harabasz"
            ]
        ].to_string(index=False)
    )



K-Means

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.5355
 3      0.4218
 5      0.3602

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          0.6611
 3          0.8413
 8          0.8946

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2        188461.2235
 3        165386.6588
 4        143820.4176


GMM

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.5288
 3      0.4165
 4      0.3433

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          0.6666
 3          0.8441
 5          1.0017

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2        183950.9758
 3        162085.3184
 4        140412.8788


Agglomerative

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.5243
 3      0.3704
 4      0.3124

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          0.6719
 5          0.9763
 8          0.9802

Top 3 Calinski-H